# 34 · The support desk as a service: FastAPI, Docker, a three-replica fleet and a load test
Level L6 · Part 6 · core · ~30 min · builds on 32

## Goal
Serve the support desk as an API in a container: a model-backed streaming route and the order and refund routes. Run
three replicas behind one gateway, load-test them, check a v2 route on a canary before it takes traffic, and roll it
back with no failed `GET`.

**Scope.** The model-backed agent is built in lessons 02–31. This lesson serves it through one route, `POST /ask/stream`,
which calls the model and streams its answer. The notebook makes **one** model call, in that route (replayed from the
recording, no key needed). The order and refund routes need no model, and the fleet and the load tests make no model
call, so they are free and repeatable.

## Why it matters
Until now the support desk ran inside one notebook process. A real desk has many customers at once and must survive a
crash. Run the cell below. It is a refund on two replicas, each with its own memory, the way two plain processes behave.
The first try lands on replica A and the retry lands on replica B.

In [1]:
replica_a, replica_b = {}, {}  # each replica keeps its own record of the refund keys it has seen


def refund_on(replica: dict, key: str, amount: float) -> str:
    if key in replica:              # the duplicate check sees only this replica's memory
        return "saved answer"
    replica[key] = amount           # ...and the money moves here
    return f"paid {amount:.2f} USD"


print(refund_on(replica_a, "refund:r-2001", 15.00))  # the first try, on replica A
print(refund_on(replica_b, "refund:r-2001", 15.00))  # the retry, on replica B: its memory is empty
paid = sum(sum(r.values()) for r in (replica_a, replica_b))
print(f"paid {paid:.2f} USD for a 15.00 USD refund")

paid 15.00 USD
paid 15.00 USD
paid 30.00 USD for a 15.00 USD refund


**What it makes easy:** the replicas share one store, so every retry sees the same keys, and any replica can answer any
request. **When you don't need it:** one user, one process, a demo on your laptop. Then a plain function call is enough.

## The idea
A service is a function that a network can call. FastAPI turns Python functions into routes and checks each input before
the code runs. Docker packs the service and its libraries into an image, so every replica is the same. A gateway (here
nginx) spreads the requests over the replicas. Replicas keep no state of their own: the state lives in a shared store
(Redis). A new version goes to one canary replica first, and it leaves the traffic in one command if it fails.

Analogy: a shop with three identical tills and one shared order book. A customer can pay at any till, and a second
attempt at the same refund finds the entry already in the book.

## Picture
```mermaid
flowchart LR
  C[Client] --> G[Gateway nginx :8340]
  G --> R1[Replica v1]
  G --> R2[Replica v1]
  G --> R3[Replica v1]
  G -->|canary: about 1 in 4| V2[Replica v2]
  R1 --> S[(Redis: keys and counters)]
  R2 --> S
  R3 --> S
  V2 --> S
```

## Before you start
Three things. **Docker** with the Compose plugin: the check cell below says whether it is here. **The environment
switch:** the Docker cells run only when you start the notebook with `TUT_ENABLE=docker,slow` (both words, comma, no
spaces). **The model:** the streaming route uses the recording shipped with the course, so no key is needed to run it.

If the check says `False`, or you did not set `TUT_ENABLE`, the Docker cells are skipped on purpose and their results
show the recorded output (labelled, in `data/recorded/34.json`). That is the normal path on a laptop, and the rest of the
lesson still runs.

In [2]:
import shutil
import subprocess

HAS_DOCKER = bool(shutil.which("docker")) and subprocess.run(
    ["docker", "compose", "version"], capture_output=True).returncode == 0
print("docker with compose:", HAS_DOCKER)

docker with compose: True


## Step by step
**Step 1 · the service as one process (no Docker).** `FastAPI` gives typed routes, input checks, streaming and a generated
API page. The store is one small module: Redis when `REDIS_URL` is set, a dictionary when it is not. The service reads
its orders from the golden set in `../data/support_eval.json` (fictional orders, written by hand for this course).
First, the helpers: imports, addresses and the golden orders.

In [3]:
import atexit, json, os, sys, threading, time
from pathlib import Path

import httpx

FLEET_DIR = Path("../apps/34/fleet")  # the service (store.py, app.py, Dockerfile), shipped with the course
GOLDEN_FILE = Path("../data/support_eval.json").resolve()  # the golden set: fictional orders and tasks
REPLICA_URL, GATEWAY = "http://127.0.0.1:8341", "http://localhost:8340"    # one local replica; the gateway port
RECORDED = json.loads(Path("../data/recorded/34.json").read_text())  # shipped with the course
RESULTS, FLEET_UP = {}, False                              # RESULTS: what the Docker cells measured
print("recorded results shipped for:", sorted(RECORDED))

recorded results shipped for: ['gate', 'load_canary', 'load_v1', 'refund', 'rollback', 'up']


**store.py (shipped in `../apps/34/fleet/`).** The two storage classes have the same three methods, so `app.py` does not care
which one it gets. `set_once` with Redis uses `SET ... NX`: an atomic "write only if absent", so when two replicas race,
the first writer wins. The file is 40 lines; the cell shows its shape.

In [4]:
print("\n".join(l for l in (FLEET_DIR / "store.py").read_text().splitlines() if l.startswith(("class ", "    async def", "def "))))

class MemoryStore:  # one process only
    async def incr(self, key):
    async def set_once(self, key, value) -> bool:  # True only for the first writer
    async def get(self, key):
class RedisStore:  # every replica shares this one
    async def incr(self, key):
    async def set_once(self, key, value) -> bool:  # SET NX: write only if absent, the first writer wins
    async def get(self, key):
def make_store():


**app.py (shipped in `../apps/34/fleet/`).** The routes. Each response carries the replica's name, its version and the
shared request count in headers. The replica name defaults to `socket.gethostname()`: inside a container that is the
container id, which is why the results show hex ids such as `ffecb897cb99`. The app does not import the model SDK. The
notebook attaches a client to `model`; a container started without one answers the model route with `503`. The routes:

In [5]:
import fastapi; print("FastAPI", fastapi.__version__)  # the framework the service is built on
print("\n".join(l for l in (FLEET_DIR / "app.py").read_text().splitlines() if l.startswith(("@app", "async def", "def ", "class "))))

FastAPI 0.154.0
@app.middleware("http")
async def stamp(request: Request, call_next):
@app.get("/health")
async def health():
@app.get("/lookup/{order_id}")
async def lookup(order_id: str):
class Ask(BaseModel):
@app.post("/ask/stream")
def ask_stream(body: Ask):
class Refund(BaseModel):
@app.post("/refunds")
async def refund(body: Refund, idempotency_key: str = Header(...)):  # FastAPI reads this as the header Idempotency-Key
@app.get("/charges")
async def charges():


**Run one replica in this notebook.** The replica is the same `app.py`, served by uvicorn on port 8341 in a background
thread. The cell waits for the health route to answer, with a timeout, instead of sleeping for a fixed time. The replica
stops at the end of the notebook, even if a later cell fails. (The container fleet further down is a separate process.)

In [6]:
import uvicorn

sys.path.insert(0, str((Path("..") / "apps" / "34").resolve()))
import fleet_tools  # ../apps/34/fleet_tools.py, shipped with the course: wait_for, Compose, the load test, the gate
os.environ.update(ORDERS_FILE=str(GOLDEN_FILE), AGENT_VERSION="v1", REPLICA_NAME="local")
sys.path.insert(0, str(FLEET_DIR))
import app as fleet_app  # the same file the image runs

server = uvicorn.Server(uvicorn.Config(fleet_app.app, host="127.0.0.1", port=8341, log_level="warning"))
replica_thread = threading.Thread(target=server.run, daemon=True)  # log_level warning: the replica is quiet here
replica_thread.start()
atexit.register(lambda: setattr(server, "should_exit", True))


import fleet_tools  # ../apps/34/fleet_tools.py, shipped with the course: wait_for, Compose, the load test, the gate
print("health:", fleet_tools.wait_for(f"{REPLICA_URL}/health"))

health: {'ok': True, 'version': 'v1'}


**Call the routes.** The lookup is a plain `GET`. An unknown order gets a `404`, not a crash.

In [7]:
with httpx.Client(base_url=REPLICA_URL, timeout=5) as c:
    print(c.get("/lookup/1002").json())
    print("unknown order:", c.get("/lookup/9999").status_code)

{'order_id': '1002', 'status': 'in transit'}
unknown order: 404


**The model route (needs-llm).** `POST /ask/stream` sends the question to the model and streams the answer back, one JSON
line per piece of text, so a client can show it before the answer is complete. This is the one model call in the
notebook. The recorder is switched on for this cell only, because it records every HTTP call made while it is on, and the
notebook's other calls go to the local replica and the gateway. The recording replays the call with no key.

In [8]:
sys.path.insert(0, "..")  # the course folder: llm_replay.py and tut_env.py live there
import llm_replay  # course helper: records and replays model calls (not part of the SDK)
from google import genai  # the official SDK; in your own project: genai.Client(...)
from tut_env import env  # course helper: reads a setting from the .env file

llm_replay.start("34_serving-and-fleet")  # the recorder covers this cell only (see the note above)
fleet_app.model = genai.Client(vertexai=True, api_key="replay" if llm_replay.is_replaying() else env("GOOGLE_CLOUD_API_KEY"),
                               http_options=genai.types.HttpOptions(timeout=30_000))  # the replica's route uses this client
with httpx.Client(base_url=REPLICA_URL, timeout=60) as c:
    with c.stream("POST", "/ask/stream", json={"question": "My parcel for order 1093 never came. Can I get my money back?"}) as r:
        events = [json.loads(line) for line in r.iter_lines() if line]
llm_replay.stop()  # saves the recording (record mode); replay mode just closes it
answer = "".join(e["text"] for e in events if e["event"] == "token")
print(len(events), "stream events from replica", events[-1]["replica"], "· answer:", answer)

LLM mode: replay (34_serving-and-fleet.yaml)
10 stream events from replica local · answer: Hello! Thank you for reaching out to the Brightside Outfitters support desk.

I would be happy to assist you with your inquiry regarding order 1093. Please note that as a support representative, I do not have direct access to your personal order history, shipping status, or payment details.

**What I can do:**
* I can provide you with our standard return and refund policy guidelines.
* I can explain the steps you need to take to initiate a formal investigation into a missing package.
* I can guide you on how to contact our billing department to request a refund review.

**What I cannot do:**
* I cannot look up the status of order 1093 or verify its delivery location.
* I cannot process a refund directly through this chat.
* I cannot see your personal account information.

To move forward with your request, please send a formal inquiry to our support team at **name@example.


**The input check happens before any money moves.** The amount `"fifteen"` is not a number. FastAPI refuses it with a
`422` and the refund code never runs.

In [9]:
with httpx.Client(base_url=REPLICA_URL) as c:
    c.post("/refunds", json={"order_id": "1011", "amount_usd": "fifteen"},
           headers={"Idempotency-Key": "refund:r-2100"}).raise_for_status()  # the 422 raises here

HTTPStatusError: Client error '422 Unprocessable Entity' for url 'http://127.0.0.1:8341/refunds'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/422

**The fix.** The same route with a number takes the refund, and the answer names the replica that paid.

In [10]:
with httpx.Client(base_url=REPLICA_URL) as c:
    print(c.post("/refunds", json={"order_id": "1011", "amount_usd": 15.0},
                 headers={"Idempotency-Key": "refund:r-2101"}).json())

{'first_try': True, 'order_id': '1011', 'amount_usd': 15.0, 'replica': 'local'}


**Step 2 · the image and the fleet (needs-docker).** The Dockerfile pins the Python image and the library versions (shown
below). The `--host 0.0.0.0` matters: inside a container, a server bound to `127.0.0.1` cannot be reached by the gateway. The
Compose file and the gateway config are provided files in `../infra/serving/`. Compose starts Redis, the v1 replicas
(`--scale agent=3` in the start cell) and the gateway. Its extra service, `agent-v2`, is the canary, started later. The
containers get no model key, so their `/ask/stream` answers `503`; this lesson's fleet checks the lookup and refund routes.

In [11]:
print((FLEET_DIR / "Dockerfile").read_text())  # pinned Python and library versions; --host 0.0.0.0 for the container

# The fleet image: the same files run as v1 or v2; compose.yml sets AGENT_VERSION.
FROM python:3.12-slim-bookworm
WORKDIR /srv
RUN pip install --no-cache-dir "fastapi==0.150.0" "uvicorn==0.54.0" "redis==8.1.0"
COPY store.py app.py ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]



**The gateway: the key lines in full.** `upstream agent_pool` is the default pool: every replica of both versions. The
`map` reads the `X-Route` header and picks a pool. A request with no `X-Route` header matches the `default` line and goes
to `agent_pool`, so it can reach the canary. nginx spreads requests over the four servers in turn (three v1, one v2):
that is the canary share, about 1 in 4. `proxy_next_upstream` retries a failed `GET` on another server. It does not retry
a `POST`, because `non_idempotent` is not set. Setting it would let a timed-out refund reach a second replica, which is
what the idempotency key guards against.

In [12]:
NGINX = (Path("../infra/serving") / "nginx.conf").read_text()  # shipped with the course
print(NGINX[NGINX.index("upstream agent_pool"):NGINX.index("    server {")].rstrip())
print("\n".join(l.strip() for l in NGINX.splitlines() if "proxy_next_upstream" in l))

upstream agent_pool {                    # default: every replica of both versions
        zone agent_pool 64k;
        server agent:8000 resolve;
        server agent-v2:8000 resolve;
    }
    upstream live_only {
        zone live_only 64k;
        server agent:8000 resolve;
    }
    upstream canary_only {
        zone canary_only 64k;
        server agent-v2:8000 resolve;
    }

    map $http_x_route $backend {
        "live"    live_only;
        "canary"  canary_only;
        default   agent_pool;
    }
proxy_next_upstream error timeout http_503;


**Helpers: the fleet's Compose calls, one load test, and a result viewer.** The helpers live in `../apps/34/fleet_tools.py`
(shipped with the course). `load_test` sends many lookups through the gateway. **p50** is the median latency (half the
requests were faster); **p95** is the latency that 95 in 100 requests beat. `show` prints a live result, or the recorded
one, labelled.

In [13]:
def show(key: str) -> None:
    """Print a live result, or the recorded one (labelled) when the fleet did not run here."""
    print(RESULTS[key] if key in RESULTS else "recorded output (the fleet did not run here): " + str(RECORDED.get(key)))


print("helpers ready")

helpers ready


**Start the fleet.** Redis, three v1 replicas and the gateway come up together. `--wait` returns when the containers are
running and Redis passes its health check. The gateway's health route is then polled until it answers. That poll calls
`/health` through the gateway, so the replicas see it too. The build installs the libraries inside the image, so the
first run is slower than the rest. If this cell is skipped, every Docker result below is shown as recorded output.

In [14]:
if HAS_DOCKER:
    fleet_tools.compose("up", "-d", "--build", "--wait", "--scale", "agent=3", "redis", "agent", "gateway")
    RESULTS["up"] = {"gateway": fleet_tools.wait_for(f"{GATEWAY}/health", timeout=120),
                     "containers": [l for l in fleet_tools.compose("ps", "--format", "{{.Service}} {{.State}}").splitlines() if l]}
    FLEET_UP = True

⏭  Skipped: this cell needs [docker,slow]. Run it yourself, or re-run with TUT_ENABLE=docker,slow.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [15]:
show("up")

recorded output (the fleet did not run here): {'gateway': {'ok': True, 'version': 'v1'}, 'containers': ['agent running', 'agent running', 'agent running', 'gateway running', 'redis running']}


**Step 3 · a load test through the gateway (needs-docker).** 300 `GET` requests with 20 at a time. The gateway spreads them
over the replicas. The shared counter (`X-Requests-Seen`) comes from Redis, so the 300 requests get 300 consecutive numbers
in one run, whichever replica answered (`one_shared_counter`). Watch the p95: the average hides the slow tail. The replica
names are container ids. A healthy run has every code at `200`.

In [16]:
if FLEET_UP:
    RESULTS["load_v1"] = fleet_tools.load_test()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [17]:
show("load_v1")

recorded output (the fleet did not run here): {'requests': 300, 'codes': {'200': 300}, 'p50_ms': 14.3, 'p95_ms': 62.3, 'replicas': 3, 'versions': {'v1': 300}, 'one_shared_counter': True}


**Step 4 · a refund retried on other replicas (needs-docker).** The same refund, with the same idempotency key, is sent
four times. The gateway spreads the tries over the replicas. The money moves once, because the key is in Redis, which all
of them share. The count of charges is read back from the store.

In [18]:
if FLEET_UP:
    posts = [fleet_tools.CLIENT.post(f"{GATEWAY}/refunds", json={"order_id": "1011", "amount_usd": 15.0},
                                     headers={"Idempotency-Key": "refund:r-2001"}) for _ in range(4)]
    RESULTS["refund"] = {"tries": [{"first_try": r.json()["first_try"], "replica": r.headers["X-Replica"]} for r in posts],
                         "charges": fleet_tools.CLIENT.get(f"{GATEWAY}/charges").json()["charges"]}

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [19]:
show("refund")

recorded output (the fleet did not run here): {'tries': [{'first_try': True, 'replica': '217049d1cc6e'}, {'first_try': False, 'replica': '6b133680015b'}, {'first_try': False, 'replica': '1a3e5c39a4f9'}, {'first_try': False, 'replica': '217049d1cc6e'}], 'charges': 1}


**Step 5 · a canary and a release gate (needs-docker).** The new version, v2, adds one field, `late`. It starts as one
replica beside the three v1 replicas, so about one request in four reaches it. Before it takes that traffic, the gate asks
both versions the same order-lookup questions from the golden set and compares the answers, as lesson 28 compared a
candidate with the live version. The `X-Route` header picks the pool: `live` for v1, `canary` for v2. Any difference fails
the gate, and the canary is stopped.

In [20]:
if FLEET_UP:
    fleet_tools.compose("up", "-d", "--wait", "agent-v2")
    fleet_tools.wait_for(f"{GATEWAY}/health", headers={"X-Route": "canary"})  # the canary answers through the gateway
    n, diffs = fleet_tools.gate(GOLDEN_FILE, GATEWAY)  # both versions answer the same golden questions
    RESULTS["gate"] = {"tasks": n, "differences": diffs, "verdict": "PASS" if not diffs else "FAIL"}
    if diffs:
        fleet_tools.compose("stop", "agent-v2")  # the gate failed: the canary leaves the pool at once

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [21]:
show("gate")

recorded output (the fleet did not run here): {'tasks': 10, 'differences': [], 'verdict': 'PASS'}


With the gate passed, the canary takes its share. The load test shows both versions answering, and every code is `200`.

In [22]:
if FLEET_UP:
    RESULTS["load_canary"] = fleet_tools.load_test()

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [23]:
show("load_canary")

recorded output (the fleet did not run here): {'requests': 300, 'codes': {'200': 300}, 'p50_ms': 14.0, 'p95_ms': 39.1, 'replicas': 4, 'versions': {'v1': 225, 'v2': 75}, 'one_shared_counter': True}


**Step 6 · roll back under load (needs-docker).** Something goes wrong with v2. The stop is a **simulated failure**: a
timer stops the canary 0.2 s into a 600-request load test, as if it had crashed. The gateway setting
`proxy_next_upstream error timeout http_503` retries each failed `GET` on another replica, so no customer should see an
error. A dead replica leaves the pool in two steps: after the failed attempt nginx marks that server as failed (for 10 s by
default), and because the server line has `resolve`, nginx drops it once Docker's DNS stops returning its address (checked
every 2 s). Refunds (`POST`) are not retried by the gateway: their safety comes from the idempotency key.

In [24]:
if FLEET_UP:
    stopper = threading.Timer(0.2, lambda: fleet_tools.compose("stop", "agent-v2"))  # the simulated failure
    stopper.start()
    RESULTS["rollback"] = fleet_tools.load_test(n=600)
    stopper.join()
    after = fleet_tools.load_test(n=60)
    RESULTS["rollback"]["after_rollback"] = {"codes": after["codes"], "versions": after["versions"]}  # no v2 now

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**Result.** Live when Docker ran here; otherwise the recorded output, labelled as such.

In [25]:
show("rollback")

recorded output (the fleet did not run here): {'requests': 600, 'codes': {'200': 600}, 'p50_ms': 17.9, 'p95_ms': 46.4, 'replicas': 4, 'versions': {'v1': 531, 'v2': 69}, 'one_shared_counter': True, 'after_rollback': {'codes': {'200': 60}, 'versions': {'v1': 60}}}


**Step 7 · tear down (needs-docker).** The fleet, its network and its volumes go, and the results are saved to
`../data/34/run.json`. The shipped `../data/recorded/34.json` holds an earlier real run, so the printouts above show
results on a machine without Docker. A live run does not overwrite it.

In [26]:
if FLEET_UP:
    fleet_tools.compose("down", "-v", "--remove-orphans")
    Path("../data/34").mkdir(parents=True, exist_ok=True)
    Path("../data/34/run.json").write_text(json.dumps(RESULTS, indent=2))
    print("fleet stopped; results saved to data/34/run.json")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


## At real scale
- **Replicas keep no state.** The Why cell showed the double refund. Shared keys make a retry safe on any replica
  (→ lesson 32).
- **Health has two questions.** Liveness says the process is alive; readiness says it can take traffic. This fleet has
  one route. Production splits them, so a slow replica is taken out without a restart.
- **Roll out in steps.** A canary is one replica here. In production it is a share of traffic held for a while, and the
  gate reads cost and error rate too (→ lesson 36).
- **Capacity is concurrency divided by latency.** 20 requests in flight at 0.5 s each give 40 per second. Plan with the
  p95 at peak; the model is most of that latency (→ lesson 33).
- **Compose is one machine.** It shows the pattern, not a cluster. Kubernetes is its own course.

## Decision guide
| Situation | Approach | Why |
|---|---|---|
| The service must survive a replica crash | several stateless replicas behind a gateway | the gateway sends the next request to a live one |
| A retry may hit another replica | shared store for idempotency keys and counters | every replica sees the same saved answer |
| A new version to test on real traffic | one canary replica and a release gate first | a bad answer reaches a few customers, not all |
| A failed canary | stop it; the gateway drops it from the pool | rollback in seconds, no image rebuild |
| More load than one process can take | more replicas, not more threads | one process has one event loop and one CPU limit |

## Common mistakes
1. **Trusting the answer without checking the status code.** An unknown order returns `404` with a `detail` message, not a
   `status`. Reading `["status"]` straight away fails with a `KeyError` that hides the real problem. Check the code first.

In [27]:
with httpx.Client(base_url=REPLICA_URL) as c:
    print(c.get("/lookup/9999").json()["status"])  # the 404 body has no "status" key

KeyError: 'status'

The fix: call `raise_for_status()` or test `status_code` before you read the body.

2. **Binding to `127.0.0.1` inside a container.** Only processes in the container can reach it, so the gateway fails.
   The fix is `--host 0.0.0.0`, as in the Dockerfile.
3. **Keeping keys in process memory.** The Why cell of this lesson paid twice. Keep state in the shared store.
4. **Retrying a POST without a key.** The gateway does not retry POST. A client that retries must reuse the same key.

## Try it
The refund key is what stops a double payment. On the local replica (port 8341), send the same refund twice with one new
key, `refund:r-2200`, for order `1011` and `10.0` USD. Keep the second answer in `second` and print it. What is
`first_try` the second time, and why?

In [28]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [29]:
with httpx.Client(base_url=REPLICA_URL) as c:
    first = c.post("/refunds", json={"order_id": "1011", "amount_usd": 10.0},
                   headers={"Idempotency-Key": "refund:r-2200"}).json()
    second = c.post("/refunds", json={"order_id": "1011", "amount_usd": 10.0},
                    headers={"Idempotency-Key": "refund:r-2200"}).json()
print("first try:", first["first_try"], "· second try:", second)
# first_try is False the second time: the replica finds the key already in its store and returns the saved answer.
# No second payment moves, and the replica and amount are the ones from the first try.

first try: True · second try: {'first_try': False, 'order_id': '1011', 'amount_usd': 10.0, 'replica': 'local'}


## Recap
- FastAPI turns functions into typed routes. Uvicorn serves them, and the same code runs as one process or in a container.
- Replicas keep no state of their own. A shared store (Redis) holds the keys and counters, so any replica can answer a retry.
- A gateway spreads traffic over the replicas. A canary gets a small share, and a release gate decides before it gets more.
- Rollback is a stop, not a rebuild. Gateway retries cover reads; the idempotency key covers the refund.

**Interview:** "How do you deploy an agent service with zero downtime?"
Strong answer: "Run stateless replicas behind a gateway with health checks and a shared store for idempotency keys. Send the
new version to one canary, run a regression gate on the golden set, then shift traffic in steps. Roll back by removing the
canary from the pool."

**Stop the local replica.** The last cell, so the Common mistakes and Try it cells above could still reach it. Setting
`should_exit` asks uvicorn to shut down, and the thread ends with it.

In [30]:
server.should_exit = True
replica_thread.join(timeout=10)